In [17]:
import pandas as pd


df_all_shuffled_training_mass = pd.read_pickle("pid_counts_table.pkl") # training sample

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf

In [18]:
# Dividing columns

X=df_all_shuffled_training_mass[['N_e', 'N_mu', 'N_K', 'N_p', 'n_tracks']].values
mass=df_all_shuffled_training_mass['m_inv_GeV'].values

In [19]:
# Train/test split

X_train, X_test, mass_train, mass_test = train_test_split(X, mass, test_size=0.05, random_state=42)

scaler = MinMaxScaler()

In [20]:
#Fitting and transforming data

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(X_train.shape, X_test.shape)



(47608, 5) (2506, 5)


In [21]:
#Debugging
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1" # Disable GPU (if any)

In [22]:
# Autoencoder

input_dim = X_train.shape[1] # should be 6
encoding_dim = 4 # latent space dimension

input_layer = Input(shape=(input_dim,))

encoded = Dense(encoding_dim, activation='relu')(input_layer)

decoded = Dense(input_dim, activation='sigmoid')(encoded)

autoencoder = Model(inputs=input_layer, outputs=decoded)

from tensorflow.keras.optimizers import Adam
autoencoder.compile(optimizer=Adam(learning_rate=0.01), loss='mse')


In [23]:
# Training Autoencoder

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

history = autoencoder.fit(X_train, X_train,
    epochs=500,
    # epochs=500,
    batch_size=256,
    # batch_size=64,
    shuffle=True,
    validation_data=(X_test, X_test),
    verbose=2,
    callbacks=[early_stopping]
)

Epoch 1/500
186/186 - 1s - 3ms/step - loss: 0.0561 - val_loss: 0.0181
Epoch 2/500
186/186 - 0s - 917us/step - loss: 0.0082 - val_loss: 0.0033
Epoch 3/500
186/186 - 0s - 832us/step - loss: 0.0028 - val_loss: 0.0020
Epoch 4/500
186/186 - 0s - 903us/step - loss: 0.0015 - val_loss: 0.0011
Epoch 5/500
186/186 - 0s - 793us/step - loss: 6.7943e-04 - val_loss: 5.8866e-04
Epoch 6/500
186/186 - 0s - 916us/step - loss: 5.6286e-04 - val_loss: 5.5030e-04
Epoch 7/500
186/186 - 0s - 807us/step - loss: 5.3649e-04 - val_loss: 5.3410e-04
Epoch 8/500
186/186 - 0s - 816us/step - loss: 5.1892e-04 - val_loss: 5.2159e-04
Epoch 9/500
186/186 - 0s - 851us/step - loss: 5.0615e-04 - val_loss: 5.1155e-04
Epoch 10/500
186/186 - 0s - 805us/step - loss: 4.9511e-04 - val_loss: 5.0289e-04
Epoch 11/500
186/186 - 0s - 787us/step - loss: 4.8658e-04 - val_loss: 4.9715e-04
Epoch 12/500
186/186 - 0s - 843us/step - loss: 4.8022e-04 - val_loss: 4.9086e-04
Epoch 13/500
186/186 - 0s - 823us/step - loss: 4.7277e-04 - val_loss: 4